In [8]:
import tkinter as tk
from tkinter import messagebox
import pandas as pd
import os

# قائمة الأعراض
symptoms = [
    "ألم حارق (Burning pain)",
    "تنميل (Numbness)",
    "وخز بالإبر (Tingling)",
    "تعب/ثقل في القدم (Fatigue in feet)",
    "إحساس غير طبيعي مثل نمل يجري (Paresthesia)",
    "ألم عند لمس شيء خفيف (Allodynia)"
]

CSV_FILE = "nss_records.csv"  # اسم ملف حفظ البيانات

class NSSApp:
    def __init__(self, root):
        self.root = root
        self.root.title("حاسبة NSS - Neuropathy Symptom Score")

        # إنشاء Canvas + Scrollbar
        canvas = tk.Canvas(root, height=400)
        scrollbar = tk.Scrollbar(root, orient="vertical", command=canvas.yview)
        self.scrollable_frame = tk.Frame(canvas)

        self.scrollable_frame.bind(
            "<Configure>",
            lambda e: canvas.configure(
                scrollregion=canvas.bbox("all")
            )
        )

        canvas.create_window((0, 0), window=self.scrollable_frame, anchor="nw")
        canvas.configure(yscrollcommand=scrollbar.set)

        canvas.pack(side="left", fill="both", expand=True)
        scrollbar.pack(side="right", fill="y")

        tk.Label(self.scrollable_frame, text="🩺 حاسبة NSS 🩺", font=("Arial", 16, "bold")).pack(pady=10)

        self.vars = []

        # واجهة لكل عرض
        for symptom in symptoms:
            frame = tk.Frame(self.scrollable_frame, padx=5, pady=5, relief=tk.RIDGE, bd=2)
            frame.pack(fill="x", padx=10, pady=5)

            tk.Label(frame, text=symptom, font=("Arial", 12)).pack(anchor="w")

            has_symptom = tk.IntVar()
            night = tk.IntVar()
            severe = tk.IntVar()

            tk.Checkbutton(frame, text="أعاني من هذا العرض", variable=has_symptom).pack(anchor="w")
            tk.Checkbutton(frame, text="يزيد بالليل", variable=night).pack(anchor="w")
            tk.Checkbutton(frame, text="ألم شديد", variable=severe).pack(anchor="w")

            self.vars.append((symptom, has_symptom, night, severe))

        # زر الحساب والحفظ
        tk.Button(self.scrollable_frame, text="احسب NSS واحفظ البيانات", font=("Arial", 12, "bold"), bg="green", fg="white",
                  command=self.calculate_and_save).pack(pady=10)

    def calculate_and_save(self):
        total_score = 0
        record = {}

        for symptom, has_symptom, night, severe in self.vars:
            record[symptom] = has_symptom.get()
            record[symptom + " - يزيد بالليل"] = night.get()
            record[symptom + " - ألم شديد"] = severe.get()

            if has_symptom.get():
                total_score += 1 + night.get() + severe.get()

        # الحد الأقصى للـ NSS = 14
        if total_score > 14:
            total_score = 14

        # التصنيف
        if total_score <= 2:
            category = "طبيعي"
        elif total_score <= 4:
            category = "خفيف"
        elif total_score <= 6:
            category = "متوسط"
        else:
            category = "شديد"

        record["NSS_score"] = total_score
        record["NSS_category"] = category

        # حفظ البيانات
        if os.path.exists(CSV_FILE):
            df = pd.read_csv(CSV_FILE)
            df = pd.concat([df, pd.DataFrame([record])], ignore_index=True)
        else:
            df = pd.DataFrame([record])

        df.to_csv(CSV_FILE, index=False)

        messagebox.showinfo("النتيجة النهائية", f"NSS Score = {total_score}\nتصنيف الحالة = {category}\n✅ تم حفظ البيانات في {CSV_FILE}")

# تشغيل التطبيق
root = tk.Tk()
app = NSSApp(root)
root.mainloop()


In [9]:
def calculate_nss(symptoms_dict):
    """
    symptoms_dict: قاموس بالشكل:
    {
        "Burning pain": {"present":1, "night":1, "severe":1},
        "Numbness": {"present":1, "night":0, "severe":0},
        ...
    }
    """
    total_score = 0
    for symptom, values in symptoms_dict.items():
        if values["present"]:
            total_score += 1
            total_score += values.get("night", 0)
            total_score += values.get("severe", 0)

    # الحد الأقصى للـ NSS = 14
    if total_score > 14:
        total_score = 14

    # التصنيف
    if total_score <= 2:
        category = "طبيعي"
    elif total_score <= 4:
        category = "خفيف"
    elif total_score <= 6:
        category = "متوسط"
    else:
        category = "شديد"

    return total_score, category

# مثال على الاستخدام
patient_symptoms = {
    "Burning pain": {"present":1, "night":1, "severe":1},
    "Numbness": {"present":1, "night":1, "severe":1},
    "Tingling": {"present":1, "night":1, "severe":1},
    "Fatigue in feet": {"present":1, "night":1, "severe":1},
    "Paresthesia": {"present":1, "night":1, "severe":1},
    "Allodynia": {"present":1, "night":1, "severe":1}
}

score, category = calculate_nss(patient_symptoms)
print(f"NSS Score = {score}")
print(f"تصنيف الحالة = {category}")


NSS Score = 14
تصنيف الحالة = شديد
